# Lezione 9: Database relazionali e SQL

**Obiettivi di apprendimento:**
- Spiegare il modello di tabelle collegate.
- Descrivere le garanzie ACID.
- Scrivere query SQL di base (`SELECT`, `WHERE`, `JOIN`) usando `sqlite3`.

**Durata stimata:** 120 min


## Per iniziare: cos'è un database relazionale?

Prima di toccare codice, pensiamo a qualcosa che già conoscete: un foglio di calcolo con più schede. Una scheda clienti, un'altra transazioni. Se volete sapere quanto ha speso un cliente specifico, dovete andare alla scheda transazioni, cercare le righe di quel cliente e sommarle a mano.

Un **database relazionale** fa esattamente questo, ma in modo automatico e a qualsiasi scala. I dati vivono in **tabelle** (righe = record, colonne = attributi), e le tabelle si **collegano** tra loro tramite un campo condiviso — di solito un identificatore. Il software che gestisce tutto questo si chiama **DBMS** (Database Management System).

Oggi costruiremo, dal vivo, un mini database con due tabelle collegate e gli faremo domande reali in SQL.

In [ ]:
import sqlite3
import pandas as pd

# ':memory:' crea il database direttamente in RAM: niente
# file esterni, perfetto per la lezione e per Colab.
conexion = sqlite3.connect(':memory:')
cursor = conexion.cursor()

# Tabella 1: clientes
cursor.execute('''
    CREATE TABLE clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        ciudad TEXT
    )
''')

# Tabella 2: transacciones, collegata a clientes tramite id_cliente
cursor.execute('''
    CREATE TABLE transacciones (
        id_transaccion INTEGER PRIMARY KEY,
        id_cliente INTEGER,
        importe REAL NOT NULL,
        FOREIGN KEY (id_cliente) REFERENCES clientes (id_cliente)
    )
''')

print("Tabelle create.")

In [ ]:
# Popoliamo le tabelle con executemany: molto più efficiente che
# fare un INSERT per riga quando ne abbiamo diverse insieme.
clientes_nuevos = [
    (1, 'Ana Torres', 'Madrid'),
    (2, 'Luis Peña', 'Bogotá'),
    (3, 'Carla Ruiz', 'Madrid'),
    (4, 'Diego Soto', 'CDMX'),
]
cursor.executemany(
    'INSERT INTO clientes (id_cliente, nombre, ciudad) VALUES (?, ?, ?)',
    clientes_nuevos
)

transacciones_nuevas = [
    (101, 1, 59.90),
    (102, 1, 120.00),
    (103, 2, 34.50),
    (104, 3, 89.99),
    (105, 3, 15.00),
    (106, 4, 200.00),
]
cursor.executemany(
    'INSERT INTO transacciones (id_transaccion, id_cliente, importe) VALUES (?, ?, ?)',
    transacciones_nuevas
)

conexion.commit()
print("Dati inseriti.")

## Prima di interrogare: le garanzie ACID

Quando abbiamo eseguito `conexion.commit()` nella cella precedente, ci siamo fidati che il database salvasse le modifiche in modo affidabile. Questo è possibile grazie alle garanzie **ACID**, il pilastro dei database relazionali:

- **A**tomicità: un'operazione viene eseguita per intero oppure non viene eseguita.
- **C**oerenza: il database passa da uno stato valido a un altro stato valido, mai a metà.
- **I**solamento: due operazioni simultanee non si pestano i piedi a vicenda.
- **D**urabilità: una volta confermata una modifica, sopravvive anche se il sistema si blocca subito dopo.

Ora sì, facciamo domande al nostro database.

In [ ]:
# WHERE: filtriamo righe secondo una condizione
resultado_where = pd.read_sql_query(
    "SELECT nombre, ciudad FROM clientes WHERE ciudad = 'Madrid'",
    conexion
)
print("Clienti di Madrid:")
print(resultado_where)

In [ ]:
# JOIN: incrociamo le due tabelle tramite il campo che condividono
resultado_join = pd.read_sql_query('''
    SELECT clientes.nombre, transacciones.importe
    FROM transacciones
    JOIN clientes ON transacciones.id_cliente = clientes.id_cliente
''', conexion)
print("Transazioni con il nome del cliente:")
print(resultado_join)

In [ ]:
# GROUP BY: raggruppiamo righe e calcoliamo un aggregato per gruppo
resultado_group = pd.read_sql_query('''
    SELECT clientes.nombre, SUM(transacciones.importe) AS gasto_total
    FROM transacciones
    JOIN clientes ON transacciones.id_cliente = clientes.id_cliente
    GROUP BY clientes.nombre
    ORDER BY gasto_total DESC
''', conexion)
print("Spesa totale per cliente:")
print(resultado_group)

## Quello che abbiamo appena visto, in una frase

Con `WHERE` filtriamo, con `JOIN` incrociamo tabelle collegate e con `GROUP BY` aggreghiamo per gruppo. Notate che non abbiamo mai scritto un ciclo: abbiamo descritto a SQL *cosa* volevamo, e il motore ha deciso *come* ottenerlo.

Questo è possibile solo perché abbiamo progettato bene la relazione tra `clientes` e `transacciones` fin dall'inizio, usando `id_cliente` come chiave condivisa (chiave esterna) — esattamente il modello di tabella relazionale di cui abbiamo parlato all'inizio della lezione.

Motori reali che usano questo stesso linguaggio SQL: **IBM Db2, SQL Server, MySQL, Oracle, PostgreSQL** (e le loro versioni "as a service" nel cloud). Gli usi più comuni sono i sistemi **OLTP** (transazioni quotidiane) e il **data warehousing/OLAP** (analisi storica) — di quest'ultimo parleremo nella lezione 11.

In [ ]:
conexion.close()

## Riepilogo della lezione

- Una tabella relazionale ha righe (record) e colonne (attributi); si collega ad altre tabelle tramite una chiave condivisa.
- I database relazionali garantiscono ACID: Atomicità, Coerenza, Isolamento, Durabilità.
- `sqlite3` ci permette di creare e interrogare un vero database relazionale in memoria, senza installare nulla.
- `SELECT ... WHERE` filtra, `JOIN` incrocia tabelle collegate, `GROUP BY` raggruppa e aggrega.
- `pandas.read_sql_query` è il modo più comodo di ottenere il risultato di una query SQL come DataFrame.

**Capitolo del manuale:** [11 — Database relazionali e SQL](../manual/11_bases_relacionales_sql_IT.md)

**Notebook di pratica:** [clase09_practica_sql_bases_relacionales_IT.ipynb](../notebooks_practica/clase09_practica_sql_bases_relacionales_IT.ipynb)